# DP3 mAb vs. tiled 12mers — ungated metric distributions

Regenerates `manuscript/figures/dp3_vs_12mer_ungated.png`: a 4×6 grid of density histograms comparing the **DP3 antibody set** vs the three **tiled-12mer** antigens (1D2K / 4WAT / 6M0J) across six per-design metrics.

**Inputs (canonical in `configs/paths.py`; local copies off-cluster):** DP3 → `metrics_cylinder_full.csv` (`METRICS_ANTIBODY`); 12mer → `metrics_12mer.csv` (`METRICS_12MER`).

**Verified:** printed medians/pass-rates reproduce the prior figure exactly (DP3 5.00/30%, 1D2K 1.85/60%, 4WAT 0.43/84%, 6M0J 2.65/47%). Antibody-clash panels are `n/a` for 12mer rows (no antibody); cylinder surrogate stands in.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# resolve inputs: prefer repo config (cluster); fall back to local copies
REPO = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
LOCAL = Path("/Users/bneff/Desktop/projects/episcaf")

def _resolve():
    try:
        sys.path.insert(0, str(REPO))
        from configs.paths import METRICS_ANTIBODY, METRICS_12MER
        if Path(METRICS_ANTIBODY).exists() and Path(METRICS_12MER).exists():
            return Path(METRICS_ANTIBODY), Path(METRICS_12MER)
    except Exception:
        pass
    return (Path(os.environ.get("EPISCAF_METRICS_ANTIBODY", LOCAL/"known_antigen/analysis/data/metrics_cylinder_full.csv")),
            Path(os.environ.get("EPISCAF_METRICS_12MER",     LOCAL/"12mer_tiling/analysis/data/metrics_12mer.csv")))

METRICS_ANTIBODY, METRICS_12MER = _resolve()
OUT = REPO/"manuscript/figures/dp3_vs_12mer_ungated.png"
print("DP3   :", METRICS_ANTIBODY); print("12mer :", METRICS_12MER)

In [ ]:
GATE = 2.5  # epitope-RMSD pass gate (Å)
COLS = [
    ("epitope_chunk_rmsd",  "Epitope\nRMSD (Å)",             "low",  True),
    ("mean_pae",            "Mean PAE\n(global)",       "low",  False),
    ("epitope_pae",         "Epitope\nPAE",         "low",  False),
    ("overall_rmsd",        "Overall\nRMSD (Å)",             "low",  False),
    ("ptm",                 "pTM",                               "high", False),
    ("af3_n_clash_res",     "AF3 clash\n(real Ab)", "low",  False),
    ("cylinder_ca_clashes", "Cylinder\nclashes",                  "low",  False),
]

In [ ]:
dp3 = pd.read_csv(METRICS_ANTIBODY, low_memory=False)
m12 = pd.read_csv(METRICS_12MER, low_memory=False)
rows = [
    ("DP3 mAb",    dp3,                      "#6e6e6e", True),
    ("1D2K 12mer", m12[m12.antigen=="1d2k"], "#1f77b4", False),
    ("4WAT 12mer", m12[m12.antigen=="4wat"], "#ff7f0e", False),
    ("6M0J 12mer", m12[m12.antigen=="6m0j"], "#2ca02c", False),
]
print({n: len(d) for n,d,_,_ in rows})

In [ ]:
xlim = {}
for key,_,_,_ in COLS:
    vals = []
    for _,df,_,has_ab in rows:
        if key=="af3_n_clash_res" and not has_ab: continue
        if key in df: vals.append(pd.to_numeric(df[key], errors="coerce").dropna().values)
    v = np.concatenate(vals) if vals else np.array([0,1])
    lo = max(0.15, np.percentile(v,0.5)) if key=="ptm" else 0
    xlim[key] = (lo, np.percentile(v, 99.5))

In [ ]:
fig, axes = plt.subplots(len(rows), len(COLS), figsize=(17, 9))
for r,(name,df,color,has_ab) in enumerate(rows):
    n = len(df)
    for c,(key,label,better,is_gate) in enumerate(COLS):
        ax = axes[r,c]
        if key=="af3_n_clash_res" and not has_ab:
            ax.text(0.5,0.5,"n/a\n(no antibody)", ha="center", va="center", style="italic",
                    color="#999", transform=ax.transAxes, fontsize=15)
            ax.set_xticks([]); ax.set_yticks([])
            [s.set_visible(False) for s in ax.spines.values()]; continue
        x = pd.to_numeric(df[key], errors="coerce").dropna().values
        lo,hi = xlim[key]
        if key in ("cylinder_ca_clashes","af3_n_clash_res"):
            edges=np.arange(np.floor(lo), np.ceil(hi)+2, 2)   # integer-aligned, width 2
        elif key=="ptm":
            edges=np.linspace(lo,hi,26)
        else:
            edges=np.linspace(lo,hi,40)
        ax.hist(np.clip(x,lo,hi), bins=edges, density=True, color=color)
        med = np.median(x); ax.axvline(med, ls="--", color="k", lw=1.2)
        ax.text(0.97,0.93,f"med {med:.2f}", ha="right", va="top", transform=ax.transAxes, fontsize=14)
        if is_gate:
            ax.axvline(GATE, ls=":", color="red", lw=1.5)
            ax.text(0.97,0.78,f"{100*(x<GATE).mean():.0f}% < {GATE}", ha="right", va="top",
                    transform=ax.transAxes, fontsize=15, color="red", fontweight="bold")
        ax.set_xlim(lo,hi); ax.set_yticks([])
        if r==0:
            ax.text(0.5,0.5,"← better" if better=="low" else "better →", ha="center", va="center",
                    transform=ax.transAxes, color="#bbb", style="italic", fontsize=14)
            ax.set_title(label, fontsize=17, fontweight="bold")
        if c==0:
            ax.set_ylabel(f"{name}\nn={n:,}", color=color, fontweight="bold",
                          rotation=0, ha="right", va="center", labelpad=42, fontsize=17)
fig.suptitle("DP3 mAb vs. tiled 12mers: ungated distributions (full design sets)\n"
             "density · dashed = median · red = 2.5 Å gate / pass rate · "
             "top row: real AF3 clash vs cylinder surrogate", fontsize=20, fontweight="bold")
fig.tight_layout(rect=[0.03,0,1,0.95])
OUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT, dpi=130, bbox_inches="tight"); print("wrote", OUT)

In [ ]:
for name,df,_,_ in rows:
    em = pd.to_numeric(df["epitope_chunk_rmsd"], errors="coerce").dropna()
    print(f"{name:11s} n={len(df):>7,}  epi_rmsd med={em.median():.2f}  %<{GATE}={100*(em<GATE).mean():.0f}%")